# Age & Gender Prediction (TensorFlow/Keras 3)
Compatible with Google Colab.

In [ ]:
!pip -q install -U tensorflow pandas scikit-learn pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 24.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print(tf.__version__)

2.21.0


## Prepare DataFrame
Your dataframe must contain columns: `img`, `age`, `gender`. Update the paths as needed.

In [ ]:
IMG_SIZE=(200,200)
BATCH_SIZE=32

def load_image(path):
    img=tf.io.read_file(path)
    img=tf.image.decode_jpeg(img, channels=3)
    img=tf.image.resize(img, IMG_SIZE)
    img=tf.cast(img, tf.float32)/255.0
    return img

def make_dataset(df, shuffle=True):
    paths=df['img'].astype(str).values
    ages=df['age'].astype('float32').values
    genders=df['gender'].astype('float32').values

    ds=tf.data.Dataset.from_tensor_slices((paths, ages, genders))

    def mapper(path, age, gender):
        image=load_image(path)
        return image, {'age_output':age, 'gender_output':gender}

    ds=ds.map(mapper, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds=ds.shuffle(len(df))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Example:
# df=pd.read_csv('/content/labels.csv')
# train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)
# train_ds=make_dataset(train_df)
# test_ds=make_dataset(test_df,shuffle=False)


In [ ]:
from tensorflow.keras import layers, Model

inp=layers.Input(shape=(200,200,3))
x=layers.Conv2D(32,3,activation='relu')(inp)
x=layers.MaxPooling2D()(x)
x=layers.Conv2D(64,3,activation='relu')(x)
x=layers.MaxPooling2D()(x)
x=layers.Conv2D(128,3,activation='relu')(x)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(128,activation='relu')(x)

age=layers.Dense(1,name='age_output')(x)
gender=layers.Dense(1,activation='sigmoid',name='gender_output')(x)

model=Model(inp,[age,gender])

model.compile(
    optimizer='adam',
    loss={'age_output':'mse','gender_output':'binary_crossentropy'},
    metrics={'age_output':['mae'],'gender_output':['accuracy']}
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200, 200,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 198, 198,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 99, 99,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 97, 97,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 48, 48,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 46, 46,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age_output (Dense)  │ (None, 1)         │        129 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender_output       │ (None, 1)         │        129 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 110,018 (429.76 KB)

 Trainable params: 110,018 (429.76 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train
# history=model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=10
# )


### Why this fixes your error

This notebook uses `tf.data.Dataset` instead of `ImageDataGenerator(..., class_mode='multi_output')`, avoiding the `output_signature`/`TypeSpec` error in TensorFlow/Keras 3.
